In [ ]:
import gc

try:
    import torch
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        print('GPU cache cleared.')
    else:
        print('CUDA not available, skip GPU cache clear.')
except Exception:
    gc.collect()
    print('Torch not available, skip GPU cache clear.')


# 第 2 课：Chunking 文本切块

上一课我们已经把最小版 RAG 主线跑通了：
- `index`
- `retrieve`
- `generate`

这一课我们进入真实 RAG 里非常关键的一步：

**Chunking，也就是文本切块。**

很多 RAG 效果不好，不一定是 embedding 或模型不行，
而是文档从一开始就切得不合理。


## 1. 为什么不能总把整篇文档直接拿去检索

如果你把一整篇长文档当成一个检索单元，通常会有几个问题：

- 文档太长，主题太杂
- query 只关心其中一小段，但相似度会被整篇内容稀释
- 最后喂给模型的上下文也会太长、太散

所以真实 RAG 通常会先把长文档切成更小的片段，再去建立索引。


## 2. Chunking 的目标是什么

一个好的 chunk，通常希望做到：

- 足够小：方便精确检索
- 也不能太碎：否则上下文信息会丢失
- 最好语义相对完整：不要在一句话中间硬切开

所以 chunking 的本质不是“机械地切文本”，而是：

**在“粒度够细”和“语义够完整”之间找平衡。**


In [ ]:
from collections import Counter
import math
import re

long_document = """
Retrieval-augmented generation, or RAG, is a way to connect language models with external knowledge. 
Instead of forcing the model to answer only from its parameters, a RAG system first looks up relevant information.

A typical RAG pipeline has multiple stages. The first stage is indexing, where documents are prepared and stored in a searchable form. 
The second stage is retrieval, where the system finds passages related to a user query. 
The third stage is generation, where the model answers using the retrieved passages as context.

Chunking is one of the most important parts of indexing. If chunks are too large, retrieval may become noisy. 
If chunks are too small, the retrieved context may lose important meaning. 
Overlap is often added between chunks so that important information is not cut too aggressively.

Evaluation in RAG is usually divided into retrieval evaluation and answer evaluation. 
Retrieval evaluation asks whether the correct passages were found. 
Answer evaluation asks whether the final response is correct, grounded, and useful.
""".strip()

print(long_document)


## 3. 最简单的切块：按固定长度切

最朴素的 chunking 方法是：
- 固定每块多少字符
- 超过就直接切开

它的优点是简单、好实现。
缺点是可能会把一句话或一个语义段落硬切断。


In [ ]:
def fixed_size_chunk(text, chunk_size=120):
    chunks = []
    for i in range(0, len(text), chunk_size):
        chunks.append(text[i:i + chunk_size])
    return chunks


fixed_chunks = fixed_size_chunk(long_document, chunk_size=120)
print('固定长度切块结果：')
for i, chunk in enumerate(fixed_chunks, start=1):
    print(f'--- chunk {i} ---')
    print(chunk)
    print()


## 4. 更自然一点：按段落切

另一种常见做法是先按段落切。

优点：
- 更符合文本原有结构
- 语义通常更完整

缺点：
- 有的段落仍然太长
- 粒度可能不够细


In [ ]:
def paragraph_chunk(text):
    paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]
    return paragraphs


paragraph_chunks = paragraph_chunk(long_document)
print('按段落切块结果：')
for i, chunk in enumerate(paragraph_chunks, start=1):
    print(f'--- paragraph chunk {i} ---')
    print(chunk)
    print()


## 5. 很常见的工程做法：滑动窗口 + overlap

真实项目里很常见的一种方案是：
- 固定窗口大小
- 但相邻 chunk 之间保留一部分 overlap

这样做的好处是：
- 既能控制 chunk 大小
- 又能减少“重要信息刚好被切断”的问题


In [ ]:
def overlapping_chunk(text, chunk_size=120, overlap=30):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        if end >= len(text):
            break
        start += chunk_size - overlap
    return chunks


overlap_chunks = overlapping_chunk(long_document, chunk_size=120, overlap=30)
print('带 overlap 的切块结果：')
for i, chunk in enumerate(overlap_chunks, start=1):
    print(f'--- overlap chunk {i} ---')
    print(chunk)
    print()


## 6. 我们继续用最小检索器来观察 chunking 的影响

这一课先不引入 embedding，仍然用上一课那种最小 bag-of-words 检索器。

这样你能更专注于看：
- 文档怎么切
- 检索结果怎么变化

而不是被模型和向量库细节分散注意力。


In [ ]:
def tokenize(text):
    return re.findall(r"[a-zA-Z]+", text.lower())


def build_vocab(chunks):
    vocab = sorted(set(token for chunk in chunks for token in tokenize(chunk)))
    return {token: i for i, token in enumerate(vocab)}


def text_to_vector(text, vocab_index):
    vec = [0.0] * len(vocab_index)
    counts = Counter(tokenize(text))
    for token, count in counts.items():
        if token in vocab_index:
            vec[vocab_index[token]] = float(count)
    return vec


def cosine_similarity(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(y * y for y in b))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)


def retrieve_from_chunks(query, chunks, top_k=2):
    vocab_index = build_vocab(chunks)
    query_vec = text_to_vector(query, vocab_index)
    scored = []
    for i, chunk in enumerate(chunks):
        chunk_vec = text_to_vector(chunk, vocab_index)
        score = cosine_similarity(query_vec, chunk_vec)
        scored.append((score, i, chunk))
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[:top_k]


In [ ]:
query = 'Why does overlap help chunking in RAG?'

print('query:', query)
print('\n固定长度切块检索结果：')
for score, idx, chunk in retrieve_from_chunks(query, fixed_chunks, top_k=2):
    print(f'score={score:.3f} | chunk={idx}')
    print(chunk)
    print()

print('段落切块检索结果：')
for score, idx, chunk in retrieve_from_chunks(query, paragraph_chunks, top_k=2):
    print(f'score={score:.3f} | chunk={idx}')
    print(chunk)
    print()

print('带 overlap 切块检索结果：')
for score, idx, chunk in retrieve_from_chunks(query, overlap_chunks, top_k=2):
    print(f'score={score:.3f} | chunk={idx}')
    print(chunk)
    print()


## 7. 你现在应该能观察到什么

通常你会发现：

- 固定长度切块容易把一句话切裂
- 段落切块语义更完整，但有时粒度不够细
- overlap 切块在很多时候会更稳一点

这也是为什么真实 RAG 项目里，chunking 往往是要反复调的，不是拍脑袋定一个数字就完事。


## 8. Chunking 不只是长度问题

在真实项目里，chunking 还会考虑很多更细的因素：

- 是否按标题 / 小节切
- 是否按句子边界切
- 是否保留表格、代码块、列表结构
- 是否给 chunk 加 metadata
- 是否根据文档类型使用不同策略

所以以后你会慢慢看到：

**chunking 其实是 RAG 工程里非常核心的一层。**


## 9. Evaluation 视角下怎么判断 chunking 好不好

chunking 的好坏，最后不是靠感觉，而是要看检索指标和问答表现。

最常见的观察方式包括：

- 命中率有没有提升
- top-k 里相关 chunk 的比例有没有更高
- 最终回答是否更 grounded
- 模型是否更少丢失关键上下文

也就是说，chunking 本质上也是可以被评估和迭代优化的。


In [ ]:
gold_keyword = 'overlap'

def simple_hit(scored_results, keyword):
    return any(keyword.lower() in chunk.lower() for _, _, chunk in scored_results)

fixed_hit = simple_hit(retrieve_from_chunks(query, fixed_chunks, top_k=2), gold_keyword)
paragraph_hit = simple_hit(retrieve_from_chunks(query, paragraph_chunks, top_k=2), gold_keyword)
overlap_hit = simple_hit(retrieve_from_chunks(query, overlap_chunks, top_k=2), gold_keyword)

print('最小 chunking evaluation 示例:')
print('fixed chunk hit =', fixed_hit)
print('paragraph chunk hit =', paragraph_hit)
print('overlap chunk hit =', overlap_hit)


## 10. 这节课最该记住什么

如果你想抓住这节课的核心主线，就记住：

- RAG 的检索对象通常不是整篇文档，而是 chunk
- chunk 切得不好，后面 embedding、检索、生成都会一起受影响
- chunking 的本质是在“粒度”和“语义完整性”之间找平衡

所以 chunking 不是前处理里的小细节，而是 RAG 效果的重要地基。


## 11. 下一课最自然学什么

学完这一课后，最自然的下一步就是：

**Embedding 与向量检索**

因为真实 RAG 大多数时候不会停留在 bag-of-words，而会进入语义检索这一层。
